# Video Swin-B: solo la cabeza, con embeddings guardados

Clasificación binaria de videos: **no explícito (0)** vs. **explícito (1)**.

| Parte | Estado |
|---|---|
| Patch embedding, stages 1–4 (24 bloques), norm final | congelados (preentrenamiento Kinetics-400) |
| **Cabeza** `fc_cls` (1024 → 1 con sigmoide, o 1024 → 2 con softmax) | **se entrena** |

**Por qué es rápido.** Como el backbone no cambia, el embedding de 1024 valores de cada miniclip (salida de la norm final
+ promedio en tiempo y espacio) es siempre el mismo. Se calcula **una sola vez**, se guarda en Drive y la cabeza se entrena
sobre esos vectores en la GPU: cada época tarda segundos.

- **Train:** se guardan dos vistas por miniclip, la original y la volteada horizontalmente. En cada paso se elige una al azar
  (`flip_prob`), lo que equivale al volteo aleatorio del entrenamiento normal.
- **Val y test:** solo se guarda la vista original, porque se evalúan sin aumentos.
- **Partición:** se usa la misma de `SPLIT_DIR` que en los otros cuadernos, así que los videos de train, val y test son idénticos
  y los resultados se pueden comparar.

Antes de empezar: **Entorno de ejecución → Cambiar tipo → GPU**. La GPU solo hace falta para extraer los embeddings;
cuando ya están guardados, la cabeza se podría entrenar incluso en CPU.

In [ ]:
# Rutas en Drive (ajústalas)
PKG_DIR    = '/content/drive/MyDrive/Video-Swin-Transformer-Colab'
DATA_DRIVE = '/content/drive/MyDrive/salida/prueba'          # salida/<conjunto>: contiene miniclips/ y meta/
DATA_LOCAL = '/content/data/prueba'                          # copia local (solo hace falta para extraer)
SPLIT_DIR  = '/content/drive/MyDrive/vswin_splits/prueba'    # misma partición que los otros cuadernos

VARIANT    = 'swin_b_k400_22k'
CACHE_DIR  = f'/content/drive/MyDrive/vswin_features/prueba/{VARIANT}'   # train.pt, val.pt, test.pt

# Partición: solo se usa la primera vez, cuando SPLIT_DIR aún no tiene los .txt
SPLIT_SEED    = 0
VAL_FRACTION  = 0.15
TEST_FRACTION = 0.15

# Entrenamiento de la cabeza
BINARY_HEAD = True          # True: 1 salida + sigmoide (BCE) | False: 2 salidas + softmax (CE)
CLASS_NAMES = ['no_explicito', 'explicito']
SEED        = 0
RUN_DIR     = f'/content/drive/MyDrive/vswin_runs/prueba_head_{"sigmoid" if BINARY_HEAD else "softmax"}_seed{SEED}'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "$PKG_DIR"
!ffmpeg -hide_banner -version | head -n 1

In [ ]:
import json, os, torch, vswin
print(vswin.describe_environment())
device = vswin.default_device()

## 1. Datos y partición fija

`get_or_create_split` lee `train.txt`, `val.txt` y `test.txt` de `SPLIT_DIR`. Solo crea la partición si esos archivos no existen.

Los miniclips (`.mp4`) se copian a disco local solo si falta algún archivo de embeddings. Si ya están todos en `CACHE_DIR`,
esta celda solo lee los `meta/*.json` desde Drive, para obtener ids y etiquetas.

In [ ]:
SPLITS = ('train', 'val', 'test')
cache_ready = all(os.path.exists(f'{CACHE_DIR}/{s}.pt') for s in SPLITS)

if cache_ready:
    data_root = DATA_DRIVE          # solo se leen meta/*.json; los .mp4 no se usan
else:
    !mkdir -p "$DATA_LOCAL" && rsync -a --info=progress2 "$DATA_DRIVE/" "$DATA_LOCAL/"
    data_root = DATA_LOCAL

records = vswin.index_dataset(data_root, labels=vswin.label_from_path({'NonPorn': 0, 'Porn': 1}))
# records = vswin.index_dataset(data_root, labels='/content/drive/MyDrive/etiquetas.csv', class_names=CLASS_NAMES)

train_rec, val_rec, test_rec = vswin.get_or_create_split(
    records, SPLIT_DIR, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SPLIT_SEED)
split_rec = {'train': train_rec, 'val': val_rec, 'test': test_rec}
for name, recs in split_rec.items():
    print(f'{name:5s}:', vswin.describe_records(recs, CLASS_NAMES))

## 2. Modelo

Se carga el backbone preentrenado y se crea una cabeza nueva. `frozen_stages=4` deja el backbone congelado y en modo `eval()`.
La norm final también queda fija, porque los embeddings se calculan después de ella.

In [ ]:
vswin.set_seed(SEED)
model = vswin.build_model(VARIANT, num_classes=2, binary=BINARY_HEAD, frozen_stages=4, device=device)
print('head:', model.cls_head.fc_cls, '| sigmoid' if model.binary else '| softmax')

## 3. Embeddings: extraer una vez y reutilizar

`get_or_extract_features`:
- Si `CACHE_DIR/<split>.pt` existe y coincide con los videos y el checkpoint, **solo lo carga** (segundos).
- Si no existe, ejecuta Swin-B sobre todos los miniclips y lo guarda. Este es el único paso lento: train cuesta el doble
  porque se calculan dos vistas.
- Si Colab se desconecta durante la extracción, al volver a ejecutar la celda continúa desde `<split>.partial.pt`.
- Si la caché no coincide con la partición o con el checkpoint, se detiene con un error. Para regenerarla usa `rebuild=True`.

Las etiquetas no se toman de la caché, sino de `records`. Así, si corriges una etiqueta, no hace falta volver a extraer.

In [ ]:
feats = {}
for name, recs in split_rec.items():
    flips = (False, True) if name == 'train' else (False,)     # train: original + volteado
    feats[name] = vswin.get_or_extract_features(
        model, recs, f'{CACHE_DIR}/{name}.pt', flips=flips,
        batch_size=8,       # miniclips por forward (T4: 8 en inferencia con AMP)
        num_workers=2,
    )

for name, fs in feats.items():
    print(f'{name:5s}', fs, fs.meta.get('extraction_seconds', ''), 's')

## 4. Entrenar la cabeza

- Optimizador AdamW con warmup lineal y luego coseno. Weight decay solo en la matriz de pesos, no en el sesgo.
- `balanced_sampling=True`: cada video pesa lo mismo, sin importar cuántos miniclips (K) tenga, y cada clase también.
- `flip_prob=0.5`: en cada paso, cada miniclip usa su vista volteada con probabilidad 0.5.
- Con `BINARY_HEAD=True` la pérdida es `BCEWithLogitsLoss` (con `pos_weight` opcional); con `False`, `CrossEntropyLoss`.
- `best.pth` se elige con val (`monitor`) y se escribe una sola vez al final, con el modelo completo y la mejor cabeza.

In [ ]:
cfg = vswin.HeadTrainConfig(
    output_dir=RUN_DIR,
    epochs=50,
    lr=1e-3,
    weight_decay=0.05,
    warmup_epochs=2,
    batch_size=256,
    balanced_sampling=True,
    flip_prob=0.5,
    label_smoothing=0.0,
    pos_weight=None,          # solo con BINARY_HEAD=True
    monitor='video_acc',      # o 'video_f1', 'video_auc', 'video_balanced_acc'
    seed=SEED,
    class_names=CLASS_NAMES,
    extra={'variant': VARIANT, 'binary_head': BINARY_HEAD, 'split_dir': SPLIT_DIR, 'cache_dir': CACHE_DIR},
)
trainer = vswin.HeadTrainer(model, feats['train'], feats['val'], cfg, device=device)
print('loss:', trainer.criterion, '| steps per epoch:', trainer.steps_per_epoch)
history = trainer.fit()

In [ ]:
import matplotlib.pyplot as plt

ep = [h['epoch'] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ep, [h['train_loss'] for h in history], label='train loss')
ax[0].set_xlabel('epoch'); ax[0].legend()
for key in ['train_clip_acc', 'clip_acc', 'video_acc', 'video_f1']:
    pts = [(h['epoch'], h[key]) for h in history if key in h]
    if pts:
        ax[1].plot(*zip(*pts), marker='o', ms=3, label=key)
ax[1].set_xlabel('epoch'); ax[1].set_ylim(0, 1); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Validación y test

Al terminar `fit()`, el modelo ya tiene la mejor cabeza. La probabilidad de un video es el promedio de p(explícito)
de sus K miniclips, y se clasifica como explícito si es mayor que 0.5. Test se evalúa una sola vez, al final.

In [ ]:
def report(metrics, rows, title):
    print(f'== {title}')
    for k, v in metrics.items():
        print(f'{k:22s} {v:.4f}')
    cm = vswin.confusion_matrix([r['label'] for r in rows], [r['pred'] for r in rows], 2)
    print('\nconfusion matrix (rows = true, cols = predicted):')
    print(f'{"":14s}{CLASS_NAMES[0]:>14s}{CLASS_NAMES[1]:>12s}')
    for i, name in enumerate(CLASS_NAMES):
        print(f'{name:14s}{cm[i, 0]:14d}{cm[i, 1]:12d}')
    wrong = sorted((r for r in rows if r['pred'] != r['label']), key=lambda r: -max(r['probs']))
    if wrong:
        print('\nmisclassified (most confident first):')
    for r in wrong[:10]:
        print(' ', r['video_id'], 'true', CLASS_NAMES[r['label']], 'p(explicito)=%.3f' % r['probs'][1], 'K=%d' % r['num_clips'])
    print()
    return cm

val_metrics, val_rows = vswin.evaluate_features(model, feats['val'])
report(val_metrics, val_rows, 'validation (best head)')

test_metrics, test_rows = vswin.evaluate_features(model, feats['test'])
cm = report(test_metrics, test_rows, 'test (best head)')

with open(f'{RUN_DIR}/test_results.json', 'w', encoding='utf-8') as f:
    json.dump({'split_dir': SPLIT_DIR, 'cache_dir': CACHE_DIR, 'seed': SEED, 'binary_head': BINARY_HEAD,
               'trained': 'head only', 'variant': VARIANT, 'best_epoch': trainer.best_epoch,
               'val_metrics': val_metrics, 'test_metrics': test_metrics,
               'confusion_matrix': cm.tolist(), 'videos': test_rows}, f, indent=1)
print('saved', f'{RUN_DIR}/test_results.json')

## 6. (Opcional) Varias semillas

Como cada entrenamiento tarda segundos, se puede repetir con varias semillas sobre la misma partición y los mismos
embeddings, y reportar la media y la desviación estándar en test. Cada semilla guarda su propia carpeta `RUN_DIR`.

In [ ]:
import numpy as np
from dataclasses import replace

SEEDS = [0, 1, 2, 3, 4]
results = []
for s in SEEDS:
    vswin.set_seed(s)
    m = vswin.build_model(VARIANT, num_classes=2, binary=BINARY_HEAD, frozen_stages=4, device=device, verbose=False)
    run_dir = RUN_DIR.rsplit('_seed', 1)[0] + f'_seed{s}'
    t = vswin.HeadTrainer(m, feats['train'], feats['val'], replace(cfg, seed=s, output_dir=run_dir), device=device)
    t.fit(verbose=False)
    tm, _ = vswin.evaluate_features(m, feats['test'])
    results.append(tm)
    print(f'seed {s}: best epoch {t.best_epoch} | test video_acc={tm["video_acc"]:.4f} video_f1={tm.get("video_f1", float("nan")):.4f}')
    del m, t
    torch.cuda.empty_cache()

for key in ['video_acc', 'video_balanced_acc', 'video_f1', 'video_auc', 'clip_acc']:
    vals = np.array([r[key] for r in results if key in r])
    if len(vals):
        print(f'{key:20s} {vals.mean():.4f} ± {vals.std(ddof=1) if len(vals) > 1 else 0:.4f}')

## 7. Usar el modelo en una sesión nueva

`best.pth` contiene el modelo completo (backbone preentrenado + cabeza entrenada), así que predice directamente desde
los `.mp4`, sin la caché.

In [ ]:
# binary debe coincidir con el usado al entrenar
detector = vswin.build_model(VARIANT, num_classes=2, binary=BINARY_HEAD,
                             pretrained=f'{RUN_DIR}/best.pth', device=device).eval()

rec = test_rec[0]
clip_dir = f'{DATA_DRIVE}/miniclips/{rec.video_id}'      # o rec.clip_paths si los datos están en disco local
pred = vswin.predict_video(detector, clip_dir, batch_size=8)
print(rec.video_id, '| true:', CLASS_NAMES[rec.label], '| predicted:', CLASS_NAMES[pred.label],
      '| p(explicito) = %.3f' % pred.probs[1])